In [35]:
"""
=============================================================================
  Plant Disease Classification - Deep Learning with TensorFlow/Keras
  Models: ResNet50, VGG16, EfficientNetB0, MobileNetV2, Custom CNN
  Dataset: New Plant Diseases Dataset (Augmented)

  FIXES IN THIS VERSION:
    1. NUM_CLASSES is auto-detected from the train folder (not hardcoded).
    2. classification_report on the test set only uses the classes that
       actually appear in the test images — avoids the "11 vs 38" crash.
    3. Filename → label parser now also handles multi-word disease names
       (Banana, Tea, etc.) using a robust fuzzy-match approach.
=============================================================================
"""

import os
import re
import glob
import json
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0, MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────────
CONFIG = {
    "TRAIN_DIR":  r"D:\Leaf dataset 2\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train",
    "VALID_DIR":  r"D:\Leaf dataset 2\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\valid",
    "TEST_DIR":   r"D:\Leaf dataset 2\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\test",
    "OUTPUT_DIR": r"D:\Leaf dataset 2\outputs",

    "IMG_SIZE":       224,
    "BATCH_SIZE":     32,
    "EPOCHS":         30,
    "LEARNING_RATE":  1e-3,
    "SEED":           42,

    # NUM_CLASSES is set automatically from the train folder below
    "MODELS_TO_RUN": {
        "CustomCNN":      True,
        "MobileNetV2":    True,
        "EfficientNetB0": True,
        "ResNet50":       True,
        "VGG16":          True,
    }
}

tf.random.set_seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

TRAIN_DIR  = CONFIG["TRAIN_DIR"]
VALID_DIR  = CONFIG["VALID_DIR"]
TEST_DIR   = CONFIG["TEST_DIR"]
OUT_DIR    = CONFIG["OUTPUT_DIR"]
IMG_SIZE   = CONFIG["IMG_SIZE"]
BATCH_SIZE = CONFIG["BATCH_SIZE"]
EPOCHS     = CONFIG["EPOCHS"]
LR         = CONFIG["LEARNING_RATE"]

os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────
#  FILENAME → CLASS LABEL PARSER
# ─────────────────────────────────────────────────────────────
def _normalise(text):
    """Lowercase, remove all non-alphanumeric chars."""
    return re.sub(r'[^a-z0-9]', '', text.lower())


def filename_to_class(filename, class_names):
    """
    Map a test-image filename to one of the known class_names.

    Examples:
        AppleCedarRust1.JPG          → Apple___Cedar_apple_rust
        BananaXamthomonas23.jpg      → Banana___Xamthomonas  (or whatever the class is)
        TomatoYellowLeafCurlVirus5   → Tomato___Tomato_Yellow_Leaf_Curl_Virus

    Strategy:
        1. Strip extension and trailing digits.
        2. Normalise both the stem and all class names.
        3. Pick the class whose normalised form best overlaps with the stem.
    """
    stem = os.path.splitext(filename)[0]
    stem = re.sub(r'\d+$', '', stem).strip()
    norm_stem = _normalise(stem)

    norm_classes = {_normalise(cn): cn for cn in class_names}

    best_class  = None
    best_score  = -1

    for norm_cn, orig_cn in norm_classes.items():
        # Longest common subsequence length as overlap score
        longer  = max(len(norm_stem), len(norm_cn))
        shorter = min(len(norm_stem), len(norm_cn))
        # Count chars in common (order-insensitive)
        common  = sum((min(norm_stem.count(c), norm_cn.count(c))
                       for c in set(norm_stem)))
        score   = common / longer if longer else 0
        if score > best_score:
            best_score  = score
            best_class  = orig_cn

    return best_class


def build_test_label_map(all_files, class_names):
    """
    Returns list of (file_path, class_index, class_name) for every test image.
    Also prints a verification table (one example per class found).
    """
    label_map = []
    seen = {}

    for fpath in all_files:
        fname    = os.path.basename(fpath)
        cls_name = filename_to_class(fname, class_names)
        cls_idx  = class_names.index(cls_name)
        label_map.append((fpath, cls_idx, cls_name))
        seen.setdefault(cls_name, fname)

    print(f"\n  📋  Filename → Class mapping ({len(seen)} unique classes in test set):")
    print(f"  {'Example filename':<45} → Mapped class")
    print("  " + "─" * 80)
    for cls_name in sorted(seen):
        print(f"  {seen[cls_name]:<45}   {cls_name}")

    return label_map


# ─────────────────────────────────────────────────────────────
#  DATA LOADERS
# ─────────────────────────────────────────────────────────────
def build_train_valid():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=True,
        seed=CONFIG["SEED"],
    )
    valid_ds = tf.keras.utils.image_dataset_from_directory(
        VALID_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False,
    )

    class_names = train_ds.class_names
    num_classes = len(class_names)
    print(f"  ✅  Classes detected : {num_classes}")
    print(f"  Train batches : {len(train_ds)}")
    print(f"  Valid batches : {len(valid_ds)}")

    augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.15),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ], name="augmentation")
    normalize = layers.Rescaling(1.0 / 255)

    def preprocess_train(x, y):
        return normalize(augmentation(x, training=True)), y

    def preprocess_eval(x, y):
        return normalize(x), y

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    valid_ds = valid_ds.map(preprocess_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

    return train_ds, valid_ds, class_names, num_classes


def build_labeled_test_ds(label_map, num_classes):
    """Build a labeled tf.data.Dataset from (path, label_idx) pairs."""
    paths   = [lm[0] for lm in label_map]
    indices = [lm[1] for lm in label_map]
    normalize = layers.Rescaling(1.0 / 255)

    def load(path, label_idx):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img   = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img   = tf.cast(img, tf.float32)
        img   = normalize(img)
        label = tf.one_hot(label_idx, depth=num_classes)
        return img, label

    test_ds = (
        tf.data.Dataset.zip((
            tf.data.Dataset.from_tensor_slices(paths),
            tf.data.Dataset.from_tensor_slices(indices),
        ))
        .map(load, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    print(f"  Test batches  : {len(test_ds)}  ({len(paths)} images)")
    return test_ds


# ─────────────────────────────────────────────────────────────
#  MODEL BUILDERS  (from scratch — weights=None)
# ─────────────────────────────────────────────────────────────
def build_custom_cnn(num_classes):
    return models.Sequential([
        layers.Conv2D(32, 3, padding="same", activation="relu",
                      input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2), layers.Dropout(0.25),

        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="CustomCNN")


def _backbone_model(backbone, num_classes, dense_units, dropout, name):
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = backbone(inputs, training=True)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(dense_units, activation="relu")(x)
    x       = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name=name)

def build_mobilenetv2(num_classes):
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
    return _backbone_model(base, num_classes, 256, 0.4, "MobileNetV2_scratch")

def build_efficientnetb0(num_classes):
    base = EfficientNetB0(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
    return _backbone_model(base, num_classes, 256, 0.4, "EfficientNetB0_scratch")

def build_resnet50(num_classes):
    base = ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
    return _backbone_model(base, num_classes, 512, 0.5, "ResNet50_scratch")

def build_vgg16(num_classes):
    base = VGG16(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None)
    return _backbone_model(base, num_classes, 512, 0.5, "VGG16_scratch")

MODEL_REGISTRY = {
    "CustomCNN":      build_custom_cnn,
    "MobileNetV2":    build_mobilenetv2,
    "EfficientNetB0": build_efficientnetb0,
    "ResNet50":       build_resnet50,
    "VGG16":          build_vgg16,
}


# ─────────────────────────────────────────────────────────────
#  CALLBACKS
# ─────────────────────────────────────────────────────────────
def get_callbacks(model_name, save_dir):
    return [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(save_dir, f"{model_name}_best.keras"),
            monitor="val_accuracy", save_best_only=True, verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_accuracy", patience=7,
            restore_best_weights=True, verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3,
            min_lr=1e-6, verbose=1,
        ),
        callbacks.CSVLogger(os.path.join(save_dir, f"{model_name}_log.csv")),
    ]


# ─────────────────────────────────────────────────────────────
#  EVALUATION
# ─────────────────────────────────────────────────────────────
def evaluate(model, dataset, class_names, model_name, save_dir, split_name="Validation"):
    """
    Evaluate model on any labeled dataset.
    For the test set, only the classes that actually appear are reported
    — this avoids the 'N classes vs 38 target_names' crash.
    """
    print(f"\n  📊  Evaluating {model_name} on {split_name} set …")
    y_true, y_pred = [], []

    for x_batch, y_batch in dataset:
        preds = model.predict(x_batch, verbose=0)
        y_pred.extend(np.argmax(preds,           axis=1))
        y_true.extend(np.argmax(y_batch.numpy(), axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Only use labels that actually appear in this split
    present_labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    present_names  = [class_names[i] for i in present_labels]

    report = classification_report(
        y_true, y_pred,
        labels=present_labels,
        target_names=present_names,
        digits=4,
        zero_division=0,
    )
    tag = split_name.lower().replace(" ", "_")
    report_path = os.path.join(save_dir, f"{model_name}_{tag}_classification_report.txt")
    with open(report_path, "w") as f:
        f.write(f"Split      : {split_name}\n")
        f.write(f"Classes    : {len(present_labels)} / {len(class_names)} "
                f"(only classes present in this split are shown)\n\n")
        f.write(report)
    print(report)

    # Confusion matrix — only for classes present in this split
    cm = confusion_matrix(y_true, y_pred, labels=present_labels)
    _plot_cm(cm, present_names, model_name, save_dir, split_name)

    accuracy = float(np.mean(y_true == y_pred))
    print(f"  ✅  {split_name} Accuracy : {accuracy:.4f}")
    return accuracy


def _plot_cm(cm, class_names_present, model_name, save_dir, split_name):
    n   = len(class_names_present)
    fig_size = max(12, n // 2)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size - 2))
    sns.heatmap(cm, annot=(n <= 40), fmt="d", cmap="Blues",
                xticklabels=class_names_present,
                yticklabels=class_names_present,
                ax=ax, linewidths=0.3)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(f"{model_name} — Confusion Matrix ({split_name})", fontsize=13)
    font_size = max(5, 9 - n // 10)
    plt.xticks(rotation=90, fontsize=font_size)
    plt.yticks(rotation=0,  fontsize=font_size)
    plt.tight_layout()
    tag = split_name.lower().replace(" ", "_")
    plt.savefig(os.path.join(save_dir, f"{model_name}_{tag}_confusion_matrix.png"), dpi=150)
    plt.close()
    print(f"  📊  {split_name} confusion matrix saved.")


# ─────────────────────────────────────────────────────────────
#  HISTORY PLOT
# ─────────────────────────────────────────────────────────────
def plot_history(history, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — Training History", fontsize=14)
    axes[0].plot(history.history["accuracy"],     label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history.history["loss"],     label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_history.png"), dpi=150)
    plt.close()
    print(f"  📈  History plot saved.")


# ─────────────────────────────────────────────────────────────
#  SAVE TEST PREDICTIONS CSV
# ─────────────────────────────────────────────────────────────
def save_test_predictions(model, test_ds, label_map, class_names, model_name, save_dir):
    all_preds, all_confs = [], []
    for x_batch, _ in test_ds:
        probs = model.predict(x_batch, verbose=0)
        all_preds.extend(np.argmax(probs, axis=1))
        all_confs.extend(np.max(probs,    axis=1))

    csv_path = os.path.join(save_dir, f"{model_name}_test_predictions.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "true_class", "predicted_class", "confidence", "correct"])
        for (fpath, true_idx, true_cls), pred_idx, conf in zip(label_map, all_preds, all_confs):
            writer.writerow([
                os.path.basename(fpath),
                true_cls,
                class_names[pred_idx],
                f"{conf:.4f}",
                "YES" if pred_idx == true_idx else "NO",
            ])
    print(f"  💾  Test predictions CSV → {csv_path}")


# ─────────────────────────────────────────────────────────────
#  COMPARISON CHART
# ─────────────────────────────────────────────────────────────
def plot_comparison(results, save_dir):
    if not results:
        return
    names     = list(results.keys())
    val_accs  = [results[n]["val_accuracy"]   for n in names]
    test_accs = [results[n]["test_accuracy"]  for n in names]
    times     = [results[n]["train_time_min"] for n in names]
    colors    = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
    x, w      = np.arange(len(names)), 0.35

    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    fig.suptitle("Model Comparison — Plant Disease Classification", fontsize=14)

    b1 = axes[0].bar(x - w/2, val_accs,  w, label="Val Accuracy",  color="#4C72B0")
    b2 = axes[0].bar(x + w/2, test_accs, w, label="Test Accuracy", color="#55A868")
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=15)
    axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1.08)
    axes[0].set_title("Val vs Test Accuracy"); axes[0].legend()
    for bar, v in zip(b1, val_accs):
        axes[0].text(bar.get_x()+bar.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontsize=8)
    for bar, v in zip(b2, test_accs):
        axes[0].text(bar.get_x()+bar.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontsize=8)

    bars2 = axes[1].bar(names, times, color=colors[:len(names)])
    axes[1].set_ylabel("Training Time (minutes)")
    axes[1].set_title("Training Time"); axes[1].set_xticklabels(names, rotation=15)
    for bar, t in zip(bars2, times):
        axes[1].text(bar.get_x()+bar.get_width()/2, t+0.3, f"{t:.1f}m", ha="center", fontsize=9)

    plt.tight_layout()
    out = os.path.join(save_dir, "model_comparison.png")
    plt.savefig(out, dpi=150); plt.close()
    print(f"\n📊  Comparison chart saved → {out}")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  Plant Disease Classification — TensorFlow/Keras")
    print("=" * 60)
    print(f"  GPU : {tf.config.list_physical_devices('GPU')}")

    # ── Train / Valid (auto-detects number of classes) ────────
    print("\n📂  Loading train & validation datasets …")
    train_ds, valid_ds, class_names, num_classes = build_train_valid()

    # ── Test: flat folder, filenames = labels ─────────────────
    print("\n📂  Loading test images & parsing filename labels …")
    extensions = ["*.jpg","*.JPG","*.jpeg","*.JPEG","*.png","*.PNG","*.bmp","*.BMP"]
    all_test_files = []
    for ext in extensions:
        all_test_files.extend(glob.glob(os.path.join(TEST_DIR, ext)))
    all_test_files = sorted(all_test_files)

    if not all_test_files:
        print("  ⚠️  No test images found — test evaluation will be skipped.")
        label_map, test_ds = [], None
    else:
        label_map = build_test_label_map(all_test_files, class_names)
        test_ds   = build_labeled_test_ds(label_map, num_classes)

    # ── Train each model ──────────────────────────────────────
    all_results = {}

    for model_name, builder_fn in MODEL_REGISTRY.items():
        if not CONFIG["MODELS_TO_RUN"].get(model_name, False):
            print(f"\n⏭   Skipping {model_name}")
            continue

        print(f"\n{'='*60}")
        print(f"  Training : {model_name}")
        print(f"{'='*60}")

        save_dir = os.path.join(OUT_DIR, model_name)
        os.makedirs(save_dir, exist_ok=True)

        model = builder_fn(num_classes)          # ← uses auto-detected count
        model.compile(
            optimizer=optimizers.Adam(learning_rate=LR),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )
        model.summary()

        t0      = time.time()
        history = model.fit(
            train_ds,
            validation_data=valid_ds,
            epochs=EPOCHS,
            callbacks=get_callbacks(model_name, save_dir),
        )
        elapsed_min = (time.time() - t0) / 60

        plot_history(history, model_name, save_dir)

        val_accuracy = evaluate(model, valid_ds, class_names,
                                model_name, save_dir, split_name="Validation")

        test_accuracy = 0.0
        if test_ds is not None:
            test_accuracy = evaluate(model, test_ds, class_names,
                                     model_name, save_dir, split_name="Test")
            save_test_predictions(model, test_ds, label_map,
                                  class_names, model_name, save_dir)

        model.save(os.path.join(save_dir, f"{model_name}_final.keras"))

        result = {
            "val_accuracy":   round(val_accuracy,  4),
            "test_accuracy":  round(test_accuracy, 4),
            "train_time_min": round(elapsed_min,   2),
            "epochs_run":     len(history.history["loss"]),
            "num_classes":    num_classes,
        }
        all_results[model_name] = result
        with open(os.path.join(save_dir, "result.json"), "w") as f:
            json.dump(result, f, indent=2)

        print(f"\n✅  {model_name} | Val: {val_accuracy:.4f} | "
              f"Test: {test_accuracy:.4f} | Time: {elapsed_min:.1f} min")

    # ── Summary ───────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  FINAL RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"{'Model':<20} {'Val Acc':>10} {'Test Acc':>10} {'Time(min)':>10} {'Epochs':>8}")
    print("-" * 62)
    for name, res in all_results.items():
        print(f"{name:<20} {res['val_accuracy']:>10.4f} {res['test_accuracy']:>10.4f} "
              f"{res['train_time_min']:>10.1f} {res['epochs_run']:>8}")

    summary_path = os.path.join(OUT_DIR, "all_results_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n💾  Summary → {summary_path}")

    plot_comparison(all_results, OUT_DIR)
    print("\n🎉  All models completed successfully!")


if __name__ == "__main__":
    main()

  Plant Disease Classification — TensorFlow/Keras
  GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

📂  Loading train & validation datasets …
Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
  ✅  Classes detected : 38
  Train batches : 2197
  Valid batches : 550

📂  Loading test images & parsing filename labels …

  📋  Filename → Class mapping (8 unique classes in test set):
  Example filename                              → Mapped class
  ────────────────────────────────────────────────────────────────────────────────
  AppleScab1.JPG                                  Apple___Apple_scab
  AppleCedarRust1.JPG                             Apple___Cedar_apple_rust
  CornCommonRust1.JPG                             Corn_(maize)___Common_rust_
  PotatoEarlyBlight1.JPG                          Potato___Early_blight
  PotatoHealthy1.JPG                              Potato___healthy
  TomatoEarlyBlight1.JPG                          Tom

KeyboardInterrupt: 